# Customer Churn Prediction in the Nigerian Telecom Sector

Nigeria's telecommunications industry is one of the fastest-growing in Africa, with over 200 million subscribers across major providers. Despite this growth, churn — the rate at which customers stop using a provider's services — remains a pressing business challenge. Providers like **MTN Nigeria**, **Airtel Nigeria**, **Glo**, and **9mobile** face fierce competition, and retaining existing customers is far more cost-effective than acquiring new ones.

According to industry reports, acquiring a new telecom customer can cost up to five times more than retaining an existing one. A churn rate reduction of even 5% can significantly boost profitability. This makes churn prediction not just an academic exercise but a genuine business priority.

As the data analyst on this project, your goal is to build machine learning models that can predict whether a customer is likely to churn, and determine which model performs better for deployment in a real-world context.

You have been provided with two datasets from a regional telecom analytics firm:

- `telecom_demographics.csv` contains customer demographic information:

| Variable             | Description                                      |
|----------------------|--------------------------------------------------|
| `customer_id`        | Unique identifier for each customer.             |
| `telecom_partner`    | The telecom provider associated with the customer (MTN, Airtel, Glo, 9mobile). |
| `gender`             | The gender of the customer.                      |
| `age`                | The age of the customer.                         |
| `state`              | The Nigerian state in which the customer is located. |
| `city`               | The city in which the customer is located.       |
| `pincode`            | The area code of the customer's location.        |
| `registration_event` | When the customer first registered with the telecom provider. |
| `num_dependents`     | The number of dependents (e.g., children) the customer has. |
| `estimated_salary`   | The customer's estimated monthly income in Naira. |

- `telecom_usage.csv` contains customer usage behaviour:

| Variable      | Description                                                  |
|---------------|--------------------------------------------------------------|
| `customer_id` | Unique identifier for each customer.                         |
| `calls_made`  | The number of calls made by the customer in the last 30 days. |
| `sms_sent`    | The number of SMS messages sent by the customer in the last 30 days. |
| `data_used`   | The amount of mobile data used by the customer (in MB).      |
| `churn`       | Binary variable: 1 = churned (left the provider), 0 = still active. |

> **Business context:** A churn prediction model that is deployed by a telecom company could be used to flag at-risk customers and trigger proactive retention campaigns — such as personalised data bonuses, loyalty rewards, or targeted customer care calls. The accuracy of the model directly affects how efficiently the business can allocate its retention budget.


In [63]:
# Import required libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

---

# Task Overview

**Core Question:** Does Logistic Regression or a Random Forest Classifier produce a higher accuracy score in predicting customer churn for a Nigerian telecom provider?

---

## Task 1 — Load, Merge, and Explore

Load `telecom_demographics.csv` and `telecom_usage.csv` into separate DataFrames, then merge them into a single DataFrame named `churn_df` using the shared `customer_id` column.

After merging:
- Calculate the **proportion of customers who have churned** (i.e., the churn rate). Think about what this tells you from a business standpoint — is it a balanced or imbalanced dataset?
- Identify all **categorical variables** in `churn_df`. These will need to be handled before modelling.

> **Real-world note:** Churn datasets are often imbalanced — far more customers stay than leave. Be mindful of this as you interpret accuracy scores later.


In [64]:
# Task 1: Load, merge, and explore
# ============================================

import pandas as pd
import numpy as np

# Load datasets
demographics = pd.read_csv("telecom_demographics_nigeria.csv")
usage = pd.read_csv("telecom_usage.csv")

# Merge datasets
churn_df = pd.merge(demographics, usage, on="customer_id")

# Display basic information
print("Shape of merged dataset:", churn_df.shape)
print("\nFirst five rows:")
display(churn_df.head())

print("\nDataset Information:")
churn_df.info()

# Calculate churn rate
churn_rate = churn_df["churn"].mean()
print(f"\nOverall Churn Rate: {churn_rate:.2%}")

# Identify categorical variables
categorical_columns = churn_df.select_dtypes(include="object").columns.tolist()

print("\nCategorical Columns:")
print(categorical_columns)

Shape of merged dataset: (6500, 14)

First five rows:


,customer_id,telecom_partner,gender,age,state,city,pincode,registration_event,num_dependents,estimated_salary,calls_made,sms_sent,data_used,churn
0,15169,Airtel Nigeria,F,26,Lagos,Abuja,667173,2020-03-16,4,648530,75,21,4532,1
1,149207,Airtel Nigeria,F,74,Kano,Port Harcourt,313997,2022-01-16,0,506057,35,38,723,1
2,148119,Airtel Nigeria,F,54,Rivers,Kano,549925,2022-01-11,2,562102,70,47,4688,1
3,187288,MTN Nigeria,M,29,Oyo,Port Harcourt,230636,2022-07-26,3,202972,95,32,10241,1
4,14016,Glo,M,45,Kaduna,Ibadan,188036,2020-03-11,4,201981,66,23,5246,1



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6500 entries, 0 to 6499
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   customer_id         6500 non-null   int64 
 1   telecom_partner     6500 non-null   object
 2   gender              6500 non-null   object
 3   age                 6500 non-null   int64 
 4   state               6500 non-null   object
 5   city                6500 non-null   object
 6   pincode             6500 non-null   int64 
 7   registration_event  6500 non-null   object
 8   num_dependents      6500 non-null   int64 
 9   estimated_salary    6500 non-null   int64 
 10  calls_made          6500 non-null   int64 
 11  sms_sent            6500 non-null   int64 
 12  data_used           6500 non-null   int64 
 13  churn               6500 non-null   int64 
dtypes: int64(9), object(5)
memory usage: 711.1+ KB

Overall Churn Rate: 20.05%

Categorical Columns:
[

---

## Task 2 — Preprocess Features

Prepare the data for machine learning by doing the following:

1. **Encode categorical features** — Convert all categorical columns (e.g., `telecom_partner`, `gender`, `state`) into numeric format using one-hot encoding or label encoding.
2. **Select features** — Drop columns that would not be useful for prediction (e.g., `customer_id`, `city`, `pincode`, `registration_event`).
3. **Scale numeric features** — Apply feature scaling to the numerical columns. Store the result in a DataFrame called `features_scaled`.
4. **Define your target** — Isolate the `churn` column as your target variable `y`.

> **Why does scaling matter?** Logistic Regression is sensitive to feature magnitude — a salary in the millions of Naira and an age in the tens would otherwise be weighted very differently. Random Forest is less affected, but scaling is still good practice for fair comparison.


In [65]:
# Task 2: Encode, select features, scale, and define target
# ============================================

from sklearn.preprocessing import StandardScaler

# Create a copy of the merged dataset
churn_processed = churn_df.copy()

# Encode categorical variables
categorical_columns = ["telecom_partner", "gender", "state"]

churn_processed = pd.get_dummies(
    churn_processed,
    columns=categorical_columns,
    drop_first=True
)

# Drop unnecessary columns
columns_to_drop = [
    "customer_id",
    "city",
    "pincode",
    "registration_event"
]

churn_processed.drop(columns=columns_to_drop, inplace=True)

# Separate features and target
X = churn_processed.drop("churn", axis=1)
y = churn_processed["churn"]

# Scale the features
scaler = StandardScaler()

features_scaled = scaler.fit_transform(X)

print("Features scaled successfully.")
print("Shape of scaled features:", features_scaled.shape)
print("Target shape:", y.shape)

Features scaled successfully.
Shape of scaled features: (6500, 37)
Target shape: (6500,)


---

## Task 3 — Train Models

Split your data into training and testing sets, then train two classification models:

1. **Split the data** into `X_train`, `X_test`, `y_train`, and `y_test` using an **80/20 split**, with `random_state=42` for reproducibility.
2. **Train a Logistic Regression model** — Use `random_state=42`. Store its predictions on the test set in `logreg_pred`.
3. **Train a Random Forest Classifier** — Use `random_state=42`. Store its predictions on the test set in `rf_pred`.

> **Real-world note:** In production, models are re-trained periodically on fresh data. The 80/20 split simulates this: training on historical data and testing on "unseen" new customers. Setting a random state ensures your results are reproducible — an important requirement in professional data science workflows.


In [66]:
# Task 3: Train-test split and model training
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    features_scaled,
    y,
    test_size=0.20,
    random_state=42
)

# Logistic Regression
logreg = LogisticRegression(random_state=42)

logreg.fit(X_train, y_train)

logreg_pred = logreg.predict(X_test)

# Random Forest
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Both models trained successfully.")

Both models trained successfully.


---

## Task 4 — Evaluate and Compare Models

Assess both models on the test data:

1. **Calculate accuracy scores** for both `logreg_pred` and `rf_pred` against `y_test`.
2. **Print a classification report** for each model to understand precision, recall, and F1-score — these give a fuller picture than accuracy alone.
3. **Assign the name of the better-performing model** (by accuracy) to a variable called `higher_accuracy`. Use exactly one of these strings: `"LogisticRegression"` or `"RandomForest"`.

> **Reflection question (no code needed):** If you were advising the telecom company on which model to deploy, would accuracy alone be enough to make the decision? Consider: what is the cost of a *false negative* (predicting a customer won't churn when they actually will)?


In [67]:
# Task 4: Model evaluation
# ============================================

from sklearn.metrics import accuracy_score, classification_report

# Logistic Regression Evaluation
logreg_accuracy = accuracy_score(y_test, logreg_pred)

print("="*60)
print("LOGISTIC REGRESSION RESULTS")
print("="*60)
print(f"Accuracy: {logreg_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, logreg_pred))

# Random Forest Evaluation
rf_accuracy = accuracy_score(y_test, rf_pred)

print("="*60)
print("RANDOM FOREST RESULTS")
print("="*60)
print(f"Accuracy: {rf_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

# Compare models
if logreg_accuracy > rf_accuracy:
    higher_accuracy = "LogisticRegression"
else:
    higher_accuracy = "RandomForest"

print("="*60)
print("BETTER PERFORMING MODEL")
print("="*60)
print("Higher Accuracy Model:", higher_accuracy)

LOGISTIC REGRESSION RESULTS
Accuracy: 0.7900

Classification Report:
              precision    recall  f1-score   support

           0       0.79      1.00      0.88      1027
           1       0.00      0.00      0.00       273

    accuracy                           0.79      1300
   macro avg       0.40      0.50      0.44      1300
weighted avg       0.62      0.79      0.70      1300

RANDOM FOREST RESULTS
Accuracy: 0.7885

Classification Report:
              precision    recall  f1-score   support

           0       0.79      1.00      0.88      1027
           1       0.00      0.00      0.00       273

    accuracy                           0.79      1300
   macro avg       0.39      0.50      0.44      1300
weighted avg       0.62      0.79      0.70      1300

BETTER PERFORMING MODEL
Higher Accuracy Model: LogisticRegression


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
